# Ticket Text Processing Learning Notebook

This notebook walks through the core NLP ideas used in the support ticket classifier.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(PROJECT_ROOT / 'data' / 'sample_tickets.csv')
df.head()

## 1. Clean Text

Cleaning makes tickets easier for a model to compare. We lowercase, remove punctuation and numbers, tokenize, remove stopwords, and lemmatize when NLTK data is available.

In [ ]:
from ml.preprocessing import clean_text

example = df.loc[0, 'ticket_text']
example, clean_text(example)

## 2. Turn Text Into Numbers With TF-IDF

TF-IDF gives useful words higher values and very common words lower values.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2))
features = vectorizer.fit_transform(df['ticket_text'])
features.shape

## 3. Train And Evaluate A Classifier

In [ ]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

x_train, x_test, y_train, y_test = train_test_split(
    df['ticket_text'], df['category'], test_size=0.25, random_state=42, stratify=df['category']
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text, ngram_range=(1, 2))),
    ('classifier', ComplementNB())
])

model.fit(x_train, y_train)
predictions = model.predict(x_test)
print(classification_report(y_test, predictions, zero_division=0))

## 4. Add Priority Logic

Priority can combine model output and business rules. For example, security and outage tickets should often be high priority.

In [ ]:
from ml.priority import infer_priority

ticket = 'A user gained access to a private workspace they should not see.'
category = model.predict([ticket])[0]
priority = infer_priority(ticket, category)
category, priority